# Module 1.1 — Retrieval Metrics: Deterministic

**Where this fits:** Module 0 laid out the taxonomy — retrieval quality is the first stage of a RAG pipeline, and it's worth measuring in isolation from generation quality, because a wrong answer can come from a broken retriever *or* a broken generator, and you want a metric that tells you which. This notebook covers the deterministic, rank-aware retrieval metrics: no LLM judge, no API key, no cost — just labeled ground truth and math. Module 1.2 covers the LLM-judged retrieval metrics (Contextual Precision/Recall/Relevancy) that don't require pre-labeled relevant chunk IDs.

_Source: adapted from `04_Agent_RAG_Eval/rag_agent_eval_langgraph_new.ipynb`, Part 1._

**Interview questions this answers:**
- *How would you evaluate retrieval quality?*
- *What do Precision@K and Recall@K tell you?*
- *When would you use MRR vs. nDCG?*

**Approach:** build a labeled eval set (query → ground-truth relevant chunk IDs), run retrieval, score with rank-aware metrics.

In [1]:
# ============ IMPORTS ============
from typing import List, Dict
import math

print("Imports OK")

Imports OK


## The four metrics

- **Precision@K** — of the K chunks retrieved, how many are actually relevant? Penalizes noise in the retrieved set.
- **Recall@K** — of all relevant chunks that exist, how many did we find in the top K? Penalizes missed relevant chunks.
- **MRR (Mean Reciprocal Rank)** — 1 / rank of the *first* relevant hit. Use when there's typically one correct/best chunk and you care how early it appears.
- **nDCG (normalized Discounted Cumulative Gain)** — use when relevance is *graded* (not binary) and multiple relevant chunks can coexist; rewards relevant results appearing higher, penalized by how much higher a *more* relevant chunk should have ranked.

In [2]:
# ============ METRIC FUNCTIONS ============
def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    # Of the K chunks retrieved, how many are actually relevant?
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / k if k > 0 else 0.0


def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    # Of all relevant chunks that exist, how many did we find in top K?
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / len(relevant) if relevant else 0.0


def mrr(retrieved: List[str], relevant: List[str]) -> float:
    # Mean Reciprocal Rank: 1/rank of the FIRST relevant hit.
    # Use when there's typically one correct/best chunk and you care how early it appears.
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0


def dcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    score = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = grades.get(doc, 0)
        score += rel / math.log2(i + 2)  # rank 1 -> log2(2)=1, avoids log2(1)=0 division
    return score


def ndcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    # Normalized DCG: use when relevance is GRADED (not binary) and multiple
    # relevant chunks can coexist. Rewards relevant results appearing higher.
    dcg = dcg_at_k(retrieved, grades, k)
    ideal_order = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_order))
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined")

Metric functions defined


### Toy eval example

Query: *"What is the time complexity of BST delete?"*
- `retrieved_ids` — what the retriever actually returned, in rank order
- `relevant_ids` — ground truth (labeled once, reused across eval runs)
- `relevance_grades` — graded relevance (0-3) for nDCG; binary membership in `relevant_ids` is enough for Precision/Recall/MRR

In [3]:
# ============ TOY EVAL EXAMPLE ============
query = "What is the time complexity of BST delete?"

retrieved_ids = ["chunk_3", "chunk_1", "chunk_7", "chunk_2", "chunk_9"]
relevant_ids = ["chunk_1", "chunk_2", "chunk_5"]  # chunk_5 exists but wasn't retrieved -> hurts recall
relevance_grades = {
    "chunk_1": 3, "chunk_2": 2, "chunk_3": 0,
    "chunk_5": 3, "chunk_7": 1, "chunk_9": 0,
}

k = 5
print(f"Precision@{k}: {precision_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"Recall@{k}:    {recall_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"MRR:           {mrr(retrieved_ids, relevant_ids):.2f}")
print(f"nDCG@{k}:       {ndcg_at_k(retrieved_ids, relevance_grades, k):.2f}")

Precision@5: 0.40
Recall@5:    0.67
MRR:           0.50
nDCG@5:       0.51


**Reading the output:**
- Precision@5 = 0.40 → 2 of the 5 retrieved chunks were relevant (`chunk_1`, `chunk_2`). 3 were noise.
- Recall@5 = 0.67 → we found 2 of the 3 relevant chunks that exist; `chunk_5` was missed entirely.
- MRR = 0.50 → the first relevant chunk (`chunk_1`) appeared at rank 2, so 1/2.
- nDCG@5 factors in that `chunk_1` (grade 3) ranking below `chunk_3` (grade 0) is a bigger penalty than a small rank swap between two similarly-graded chunks would be.

## Summary

- All four metrics need **pre-labeled ground truth** (`relevant_ids` / `relevance_grades`) — that's what makes them deterministic and cheap (no LLM judge call) but also what limits them to curated eval sets rather than arbitrary production traffic.
- **Precision** catches noise, **Recall** catches misses, **MRR** rewards an early single best hit, **nDCG** rewards good ranking under graded, multi-chunk relevance.
- Next: [Module 1.2](01_Retrieval_Metrics_LLM_Judged.ipynb) covers the referenceless, LLM-judged counterparts (`ContextualPrecisionMetric`, `ContextualRecallMetric`, `ContextualRelevancyMetric`) — useful exactly when you *don't* have pre-labeled relevant chunk IDs for every query.